# Data Exploration: GEE Assets & Historical Fire Data

**Objective**: Verify data availability and visualize historical fire patterns.

**Steps**:
1. Connect to Google Earth Engine
2. Load Bichua forest beats asset
3. Visualize MODIS burnt areas (2018-2024)
4. Check Sentinel-2/HLS availability
5. Explore 2020 lockdown year (low fire baseline)

In [2]:
import ee
import sys
sys.path.append('..')
from config import *

# Initialize GEE
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"✓ Connected to GEE project: {GEE_PROJECT_ID}")
except:
    print("Authenticating...")
    ee.Authenticate()
    ee.Initialize(project='van-suraksha-alert')

✓ Connected to GEE project: van-suraksha-alert


## 1. Load Forest Beats Asset

In [3]:
# Load asset
beats = ee.FeatureCollection(BEATS_ASSET_ID)
count = beats.size().getInfo()
print(f"✓ Loaded {count} forest beats")

# Get bounds
bounds = beats.geometry().bounds().getInfo()['coordinates'][0]
print(f"AOI Bounds: {bounds}")

✓ Loaded 50 forest beats
AOI Bounds: [[78.97430934292693, 21.68665067395838], [79.12466133710824, 21.68665067395838], [79.12466133710824, 21.796625469782516], [78.97430934292693, 21.796625469782516], [78.97430934292693, 21.68665067395838]]


## 2. Historical Fire Analysis (MODIS)

In [4]:
import folium

# Load MODIS burnt area
fires = ee.ImageCollection(FIRE_DATASET)\
    .filterDate(f'{TRAINING_START_YEAR}-01-01', f'{TRAINING_END_YEAR}-12-31')\
    .select('BurnDate')\
    .filterBounds(beats.geometry())

# Count total burns
total_burns = fires.map(lambda img: img.gt(0)).sum()

# Get center for map
center = beats.geometry().centroid().coordinates().getInfo()[::-1]

# Create map
m = folium.Map(location=center, zoom_start=10)

# Add burnt areas (red = more frequent)
map_id = total_burns.getMapId({'min': 0, 'max': 10, 'palette': ['white', 'yellow', 'orange', 'red']})
folium.TileLayer(
    tiles=map_id['tile_fetcher'].url_format,
    attr='MODIS Burnt Areas',
    overlay=True,
    name='Fire Frequency (2018-2023)'
).add_to(m)

# Add beats outline
beats_style = {'color': 'blue', 'fillColor': 'transparent', 'weight': 2}
folium.GeoJson(beats.getInfo(), style_function=lambda x: beats_style, name='Forest Beats').add_to(m)

folium.LayerControl().add_to(m)
m

ModuleNotFoundError: No module named 'folium'

## 3. Lockdown Year Analysis (2020)

In [ ]:
# Compare 2019, 2020, 2021
years = [2019, 2020, 2021]
burn_counts = {}

for year in years:
    year_fires = ee.ImageCollection(FIRE_DATASET)\
        .filterDate(f'{year}-01-01', f'{year}-12-31')\
        .select('BurnDate')\
        .filterBounds(beats.geometry())
    
    burn_area = year_fires.map(lambda img: img.gt(0)).sum()
    pixel_count = burn_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=beats.geometry(),
        scale=500,
        maxPixels=1e9
    ).getInfo()['BurnDate']
    
    burn_counts[year] = pixel_count
    print(f"{year}: {pixel_count:.0f} burnt pixels")

print(f"\n2020 vs 2019: {(burn_counts[2020]/burn_counts[2019] - 1)*100:.1f}% change")
print(f"Expected: Significant decrease (lockdown effect)")

## 4. Sentinel-2/HLS Availability Check

In [ ]:
# Check HLS (Harmonized Landsat Sentinel) collection
# Note: HLS might not be directly in GEE. We'll use Sentinel-2 directly.

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
    .filterDate('2024-01-01', '2024-12-31')\
    .filterBounds(beats.geometry())\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

s2_count = s2.size().getInfo()
print(f"✓ Sentinel-2 scenes in 2024 (cloud < 30%): {s2_count}")
print(f"  ~{s2_count/12:.1f} scenes/month (need >2/month for good temporal resolution)")

if s2_count < 24:
    print("⚠️  Warning: Low scene count. May need to relax cloud cover threshold.")

## 5. Sample Data Export (for testing)

Export a small sample to verify the full pipeline.

In [ ]:
# Export a single beat's data for testing
sample_beat = beats.first().geometry()

# Get 6 months of S2 data (Jan-June 2024)
sample_s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
    .filterDate('2024-01-01', '2024-06-30')\
    .filterBounds(sample_beat)\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))\
    .select(PRITHVI_INPUT_BANDS)

print(f"Sample beat scenes: {sample_s2.size().getInfo()}")
print("Ready for training data pipeline development.")

## Summary

**Data Validation Checklist**:
- [ ] Forest beats loaded successfully
- [ ] MODIS burnt areas show clear patterns
- [ ] 2020 lockdown year shows reduced fires
- [ ] Sufficient Sentinel-2 temporal coverage
- [ ] Sample data export works

**Next Steps**:
1. Build training dataset generator
2. Download Prithvi-EO model
3. Create data loader